[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/structural-break/quickstarters/baseline/baseline.ipynb)
[![Open In Kaggle](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-kaggle.svg)](https://www.kaggle.com/code/crunchdao/structural-break-baseline)

In [1]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break ycgY7darheNPiFvSUaSrq2Bp

crunch-cli, version 8.0.0
delete /content/.crunchdao
main.py: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/30828/main.py (22131 bytes)
notebook.ipynb: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/30828/notebook.ipynb (103151 bytes)
requirements.txt: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/submissions/30828/requirements.original.txt (187 bytes)
data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_test.reduced.parquet: dow

In [2]:
! pip install pyod

In [3]:
import os
import json
import numpy as np
import pandas as pd

from joblib import dump, load
import sklearn

# Base ML
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# Gradient Boosting
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# PyOD detectors
from pyod.models.knn import KNN
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ecod import ECOD


In [4]:
import crunch

# Load the Crunch Toolings
crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 8.0.0
available ram: 12.67 gb
available cpu: 2 core
----


In [5]:
# Load the data simply
X_train, y_train, X_test = crunch.load_data()

data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match


In [6]:
X_train

value  period
id    time                  
0     0    -0.005564       0
      1     0.003705       0
      2     0.013164       0
      3     0.007151       0
      4    -0.009979       0
...              ...     ...
10000 2134  0.001137       1
      2135  0.003526       1
      2136  0.000687       1
      2137  0.001640       1
      2138  0.001074       1

[23715734 rows x 2 columns]

In [7]:
y_train

,structural_breakpoint
id,
0,False
1,False
2,True
3,False
4,False
...,...
9996,False
9997,False
9998,False


In [8]:
print("Number of datasets:", len(X_test))

Number of datasets: 101


In [9]:
X_test[0]

value  period
id    time                  
10001 0     0.010753       0
      1    -0.031915       0
      2    -0.010989       0
      3    -0.011111       0
      4     0.011236       0
...              ...     ...
      2774 -0.013937       1
      2775 -0.015649       1
      2776 -0.009744       1
      2777  0.025375       1
      2778 -0.001515       1

[2779 rows x 2 columns]

In [10]:
def _ensure_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure ('id','time') exist; if DataFrame index is MultiIndex, reset it.
    Return dataframe sorted by ('id','time').
    """
    if {"id", "time"}.issubset(df.columns):
        x = df.copy()
    else:
        x = df.reset_index()
    return x.sort_values(["id", "time"])


def _series_channels(df_one: pd.DataFrame) -> dict:
    """
    Build two 1-D channels for PyOD:
      - 'val': raw values
      - 'diff': within-period first difference (no cross-boundary leakage)
    Return dict:
      {
        'val': (X_pre, X_post),  # both are 2D arrays (n,1)
        'diff': (X_pre_diff, X_post_diff)
      }
    """
    v = df_one["value"].astype(float).values
    p = df_one["period"].astype(int).values
    v_pre, v_post = v[p == 0], v[p == 1]

    # value channel
    X_pre_val = v_pre.reshape(-1, 1)
    X_post_val = v_post.reshape(-1, 1)

    # diff channel computed separately within each segment
    def safe_diff(a):
        return np.diff(a) if a.size >= 2 else np.array([], dtype=float)

    d_pre = safe_diff(v_pre)
    d_post = safe_diff(v_post)

    X_pre_diff = d_pre.reshape(-1, 1)
    X_post_diff = d_post.reshape(-1, 1)

    return {
        "val": (X_pre_val, X_post_val),
        "diff": (X_pre_diff, X_post_diff),
    }


def _build_detectors(random_state: int) -> dict:
    """
    Build fresh PyOD detectors (one-time-use per series/channel).
    All deterministic where applicable.
    """
    return {
        "knn": KNN(n_neighbors=10, method="largest"),
        "lof": LOF(n_neighbors=20, contamination="auto"),
        "iforest": IForest(
            n_estimators=200, max_samples="auto", contamination="auto", random_state=random_state
        ),
        "ecod": ECOD(),
    }


def _aggregate_scores(scores: np.ndarray) -> dict:
    """
    Given a 1-D array of anomaly scores on POST, return multi-view aggregations:
      - mean, max, p95, slope (linear trend of scores over time index)
    """
    if scores.size == 0 or not np.isfinite(scores).any():
        return {"mean": 0.0, "max": 0.0, "p95": 0.0, "slope": 0.0}

    s = scores[np.isfinite(scores)]
    if s.size == 0:
        return {"mean": 0.0, "max": 0.0, "p95": 0.0, "slope": 0.0}

    # basic aggregations
    mean_val = float(np.mean(s))
    max_val = float(np.max(s))
    p95_val = float(np.percentile(s, 95))

    # slope via simple linear regression on index (normalize index to [-0.5, 0.5] for stability)
    idx = np.arange(len(scores), dtype=float)
    if idx.size >= 2:
        x = idx.copy()
        x = (x - x.mean()) / (x.std() + 1e-12)
        # slope of least squares fit
        slope = float(np.dot(x, scores) / (np.dot(x, x) + 1e-12))
    else:
        slope = 0.0

    # guard NaNs/infs
    def clean(v):
        return float(v) if np.isfinite(v) else 0.0

    return {
        "mean": clean(mean_val),
        "max": clean(max_val),
        "p95": clean(p95_val),
        "slope": clean(slope),
    }


def _anomaly_features_one_id(df_one: pd.DataFrame, random_state: int) -> dict:
    """
    For a single time series:
      - Build two channels: raw values, within-period diffs
      - For each channel and each PyOD detector:
          * fit on PRE
          * decision_function on POST
          * aggregate into mean/max/p95/slope
      - Return a flat feature dict
    """
    channels = _series_channels(df_one)
    feats = {}

    for ch_name, (X_pre, X_post) in channels.items():
        # short-circuit if no data
        if X_pre.shape[0] < 5 or X_post.shape[0] < 1:
            # still create the same feature names with zeros
            for det in ["knn", "lof", "iforest", "ecod"]:
                for stat in ["mean", "max", "p95", "slope"]:
                    feats[f"{ch_name}__{det}__{stat}"] = 0.0
            continue

        dets = _build_detectors(random_state)
        for det_name, det in dets.items():
            try:
                det.fit(X_pre)
                scores_post = det.decision_function(X_post)
                agg = _aggregate_scores(scores_post)
                for stat_name, stat_val in agg.items():
                    feats[f"{ch_name}__{det_name}__{stat_name}"] = stat_val
            except Exception:
                # robust fallback: fill zeros if detector fails
                for stat in ["mean", "max", "p95", "slope"]:
                    feats[f"{ch_name}__{det_name}__{stat}"] = 0.0

    # ensure plain floats
    for k, v in list(feats.items()):
        if not (isinstance(v, (int, float)) and np.isfinite(v)):
            feats[k] = 0.0

    return feats


def _build_feature_matrix(X: pd.DataFrame, random_state: int) -> pd.DataFrame:
    """
    Build per-id anomaly feature matrix for a MultiIndex (id,time) frame.
    """
    assert isinstance(X.index, pd.MultiIndex), "X must be a MultiIndex with levels ('id','time')"
    rows = []
    ids = []
    for gid, df_one in X.groupby(level=0, sort=False):
        feats = _anomaly_features_one_id(df_one, random_state=random_state)
        rows.append(feats)
        ids.append(gid)
    F = pd.DataFrame(rows, index=pd.Index(ids, name="id")).fillna(0.0)
    return F


def _oof_cv_model(model, X: pd.DataFrame, y: pd.Series, groups, n_splits: int, name: str):
    oof = np.zeros(len(X), dtype=float)
    models = []
    gkf = GroupKFold(n_splits=n_splits)

    for fold, (tr, va) in enumerate(gkf.split(X, y, groups=groups), 1):
        m = clone(model)   # ✅ safe clone
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        oof[va] = p
        auc = roc_auc_score(y.iloc[va], p)
        print(f"[{name}] Fold {fold} AUC = {auc:.6f}")
        models.append(m)

    print(f"[{name}] OOF AUC = {roc_auc_score(y, oof):.6f}")
    return oof, models


In [11]:
def train(X_train: pd.DataFrame, y_train: pd.Series, model_directory_path: str):
    """
    Train-time pipeline:
      1) Normalize input format and build deterministic config.
      2) Extract anomaly-based features (multi-model, multi-aggregation, value & diff channels).
      3) Train base learners (LogReg, LightGBM, XGBoost) with GroupKFold to get OOF.
      4) Train a meta learner (LogReg) with isotonic calibration on the OOF stack.
      5) Persist all artifacts needed for inference.
    """
    os.makedirs(model_directory_path, exist_ok=True)

    # ---- fixed deterministic config (embedded; not a global) ----
    cfg = {
        "random_state": 2024,
        "n_splits": 5,
        "lr_C": 1.0,
        "lr_max_iter": 2000,
        "lgb_params": {
            "n_estimators": 600,
            "learning_rate": 0.03,
            "num_leaves": 31,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 2.0,
            "random_state": 2024,
            "n_jobs": 1,                 # for strict determinism
            "class_weight": "balanced",
        },
        "xgb_params": {
            "n_estimators": 800,
            "learning_rate": 0.03,
            "max_depth": 5,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_lambda": 2.0,
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "n_jobs": 1,                 # for strict determinism
            "tree_method": "hist",
            "random_state": 2024,
        },
    }

    # ---- ensure input format ----
    Xt = _ensure_columns(X_train).set_index(["id", "time"]).sort_index()

    # ---- labels (Series[int] indexed by id) ----
    if isinstance(y_train, pd.DataFrame):
        if "structural_breakpoint" in y_train.columns:
            y = y_train["structural_breakpoint"].astype(int)
        else:
            y = y_train.iloc[:, 0].astype(int)
    else:
        y = y_train.astype(int)

    # ---- build feature matrix ----
    F = _build_feature_matrix(Xt, random_state=cfg["random_state"])
    # align y to F
    y = y.reindex(F.index).astype(int)

    # ---- base learners ----
    groups = F.index.values

    # 1) LogisticRegression with StandardScaler
    logreg = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        LogisticRegression(max_iter=cfg["lr_max_iter"], C=cfg["lr_C"], penalty="l2", random_state=cfg["random_state"]),
    )
    oof_lr, models_lr = _oof_cv_model(logreg, F, y, groups, n_splits=cfg["n_splits"], name="LogReg")

    # 2) LightGBM
    lgbm = LGBMClassifier(**cfg["lgb_params"])
    oof_lgb, models_lgb = _oof_cv_model(lgbm, F, y, groups, n_splits=cfg["n_splits"], name="LightGBM")

    # 3) XGBoost
    xgb = XGBClassifier(**cfg["xgb_params"])
    oof_xgb, models_xgb = _oof_cv_model(xgb, F, y, groups, n_splits=cfg["n_splits"], name="XGBoost")

    # ---- stack features (OOF) ----
    stack_cols = ["logreg", "lgbm", "xgb"]
    S = np.column_stack([oof_lr, oof_lgb, oof_xgb])
    S_df = pd.DataFrame(S, index=F.index, columns=stack_cols)

    # ---- meta model + isotonic calibration ----
    meta_base = LogisticRegression(max_iter=cfg["lr_max_iter"], C=cfg["lr_C"], random_state=cfg["random_state"])
    calibrated = CalibratedClassifierCV(estimator=meta_base, method="isotonic", cv=cfg["n_splits"])
    calibrated.fit(S_df, y)
    try:
        print(f"[META] OOF AUC = {roc_auc_score(y, calibrated.predict_proba(S_df)[:, 1]):.6f}")
    except Exception:
        pass

    # ---- persist artifacts ----
    # training feature columns (to enforce exact order at inference)
    with open(os.path.join(model_directory_path, "feature_columns.json"), "w") as f:
        json.dump(list(F.columns), f)

    # base fold models
    dump(models_lr,  os.path.join(model_directory_path, "models_logreg.joblib"))
    dump(models_lgb, os.path.join(model_directory_path, "models_lgb.joblib"))
    dump(models_xgb, os.path.join(model_directory_path, "models_xgb.joblib"))

    # meta calibrator and stack col names
    dump(calibrated, os.path.join(model_directory_path, "meta_calibrated.joblib"))
    with open(os.path.join(model_directory_path, "stack_cols.json"), "w") as f:
        json.dump(stack_cols, f)

    # persist config
    with open(os.path.join(model_directory_path, "config.json"), "w") as f:
        json.dump(cfg, f)

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

**Important workflow:**
1. Load your model;
2. Use the `yield` statement to signal readiness to the runner;
3. Process each dataset one by one within the for loop;
4. For each dataset, use `yield prediction` to return your prediction.

**Note:** The datasets can only be iterated once!

In [12]:
def infer(X_test, model_directory_path: str):
    """
    Sequential inference:
      - Load artifacts and yield once to signal readiness.
      - For each dataset:
          * build anomaly features for its id(s)
          * align to training columns
          * get base predictions (fold-avg) → stack → calibrated meta prob
          * if multiple ids in one frame (rare), average to a single float
      - yield one float per dataset
    """
    # load artifacts
    with open(os.path.join(model_directory_path, "config.json"), "r") as f:
        cfg = json.load(f)
    with open(os.path.join(model_directory_path, "feature_columns.json"), "r") as f:
        feature_columns = json.load(f)
    with open(os.path.join(model_directory_path, "stack_cols.json"), "r") as f:
        stack_cols = json.load(f)

    models_lr  = load(os.path.join(model_directory_path, "models_logreg.joblib"))
    models_lgb = load(os.path.join(model_directory_path, "models_lgb.joblib"))
    models_xgb = load(os.path.join(model_directory_path, "models_xgb.joblib"))
    calibrated = load(os.path.join(model_directory_path, "meta_calibrated.joblib"))

    # signal ready
    yield

    # iterate test datasets
    for dataset in X_test:
        Xi = _ensure_columns(dataset).set_index(["id", "time"]).sort_index()

        # build features
        Ft = _build_feature_matrix(Xi, random_state=int(cfg["random_state"]))

        # strictly align columns to training order; fill missing with 0.0
        for c in feature_columns:
            if c not in Ft.columns:
                Ft[c] = 0.0
        Ft = Ft[feature_columns].fillna(0.0)

        # base predictions (fold-averaged)
        def _avg_proba(models_list, Xdf):
            probs = []
            for m in models_list:
                probs.append(m.predict_proba(Xdf)[:, 1])
            return np.mean(np.vstack(probs), axis=0)

        p_lr  = _avg_proba(models_lr,  Ft)
        p_lgb = _avg_proba(models_lgb, Ft)
        p_xgb = _avg_proba(models_xgb, Ft)

        S_te = np.column_stack([p_lr, p_lgb, p_xgb])
        S_te_df = pd.DataFrame(S_te, index=Ft.index, columns=stack_cols)

        # calibrated meta prob per id
        p_meta = calibrated.predict_proba(S_te_df)[:, 1]

        # if a dataset unexpectedly contains multiple ids, average them to one score
        pred = float(np.mean(p_meta)) if p_meta.size else 0.5
        yield float(np.clip(pred, 0.0, 1.0))

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [13]:
crunch.test(
    # Uncomment to disable the train
    # force_first_train=False,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

22:04:38 
22:04:38 started
22:04:38 running local test
22:04:38 internet access isn't restricted, no check will be done
22:04:38 
22:04:40 starting unstructured loop...
22:04:40 executing - command=train


data/X_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_train.parquet (204327238 bytes)
data/X_train.parquet: already exists, file length match
data/X_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/X_test.reduced.parquet (2380918 bytes)
data/X_test.reduced.parquet: already exists, file length match
data/y_train.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_train.parquet (61003 bytes)
data/y_train.parquet: already exists, file length match
data/y_test.reduced.parquet: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/146/y_test.reduced.parquet (2655 bytes)
data/y_test.reduced.parquet: already exists, file length match


00:50:12 duration - time=02:45:33
00:50:12 memory - before="962.63 MB" after="2.2 GB" consumed="1.24 GB"


TypeError: Pipeline.__init__() got an unexpected keyword argument 'standardscaler'

## Results

Once the local tester is done, you can preview the result stored in `data/prediction.parquet`.

In [ ]:
prediction = pd.read_parquet("data/prediction.parquet")
prediction

### Local scoring

You can call the function that the system uses to estimate your score locally.

In [ ]:
# Load the targets
target = pd.read_parquet("data/y_test.reduced.parquet")["structural_breakpoint"]

# Call the scoring function
sklearn.metrics.roc_auc_score(
    target,
    prediction,
)